# Pakistani Fashion Intelligence
## Step 2 — Data Cleaning and Feature Engineering

**Project:** Fashion Brand Competitive Price Analytics  
**Prepared by:** Aqib Hanif  
**Assigned By:** Mam Sumayyea Salahuddin  
**Institute:** Arfa Karim Incubation Center, Peshawar  

### Purpose of this notebook

In Step 1, I collected product data from the official websites of **J., Maria.B, and Sana Safinaz**.

In this notebook, I clean that raw dataset and prepare the fields that I will need later for:

- PostgreSQL analysis
- PivotTables
- the interactive Excel dashboard
- business insights

I am keeping the steps simple so I can clearly explain what I did during my class presentation.

## 1. Import libraries

I mainly use **pandas** for data cleaning and calculations. I also use **NumPy** for a few conditional calculations.

In [1]:
# I import the libraries that I need for cleaning and analysis.

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load my raw web-scraped dataset

I use the CSV created in Step 1.

The normal project file name is `01_raw_fashion_products.csv`. I also added a small fallback for the downloaded copy that contains `(1)` in its name.

In [2]:
# I set the input and output file names for this step.

INPUT_FILE = Path("01_raw_fashion_products.csv")

# This fallback helps if my downloaded file has "(1)" in the file name.
if not INPUT_FILE.exists():
    INPUT_FILE = Path("01_raw_fashion_products(1).csv")

OUTPUT_FILE = Path("02_processed_fashion_products.csv")

# I load the raw product data into a pandas DataFrame.
df = pd.read_csv(INPUT_FILE)

print("Raw dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Raw dataset loaded successfully.
Rows: 281
Columns: 17


## 3. First look at the data

Before changing anything, I check the first few rows, column names, brands, and categories. This helps me understand whether the scraping result looks reasonable.

In [3]:
# I preview the first five rows of my raw dataset.

df.head()

,Product_ID,Snapshot_Date,Brand,Product_Name,SKU,Category,Subcategory,Collection,Original_Price,Sale_Price,Discount_Amount,Discount_Percent,Availability,Currency,Product_URL,Source_Website,Scrape_Date
0,J-001,2026-08-31,J.,Choco Vanille,PU300268-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/choco-v...,www.junaidjamshed.com,2026-08-31
1,J-002,2026-08-31,J.,Galaxy Grape,PU300267-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/galaxy-...,www.junaidjamshed.com,2026-08-31
2,J-003,2026-08-31,J.,Matcha Matcha,PU300266-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/matcha-...,www.junaidjamshed.com,2026-08-31
3,J-004,2026-08-31,J.,Pink Sugar,PU300265-DFT-100ML-REG,Unstitched,Other,Women Unstitched,6900.0,6900.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/pink-su...,www.junaidjamshed.com,2026-08-31
4,J-005,2026-08-31,J.,ENIGMA NOIR,PL300018-DFT-100ML-REG,Unstitched,Other,Women Unstitched,4500.0,4500.0,0.0,0.0,Available,PKR,https://www.junaidjamshed.com/products/enigma-...,www.junaidjamshed.com,2026-08-31


In [4]:
# I check how many products I collected from each brand.

brand_count = (
    df["Brand"]
    .value_counts()
    .rename_axis("Brand")
    .reset_index(name="Products")
)

brand_count

,Brand,Products
0,J.,100
1,Maria.B,100
2,Sana Safinaz,81


In [5]:
# I check the main categories available in my dataset.

category_count = (
    df["Category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Products")
)

category_count

,Category,Products
0,Unstitched,84
1,Formal,63
2,Pret,59
3,Luxury Pret,50
4,Stitched,25


## 4. Data quality checks

I check four basic problems:

1. missing values,
2. duplicate product IDs,
3. duplicate product URLs,
4. invalid prices.

I prefer checking these issues before creating new analytical fields.

In [6]:
# I check missing values in every column.

missing_values = (
    df.isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

missing_values

,Missing_Values
Product_ID,0
Snapshot_Date,0
Brand,0
Product_Name,0
SKU,0
Category,0
Subcategory,0
Collection,0
Original_Price,0
Sale_Price,0


In [7]:
# I check duplicate IDs and duplicate product URLs.

duplicate_ids = df["Product_ID"].duplicated().sum()
duplicate_urls = df.duplicated(subset=["Brand", "Product_URL"]).sum()

print("Duplicate Product_ID values:", duplicate_ids)
print("Duplicate product URLs:", duplicate_urls)

Duplicate Product_ID values: 0
Duplicate product URLs: 0


In [8]:
# I check for impossible or suspicious price relationships.

invalid_original_price = (df["Original_Price"] <= 0).sum()
invalid_sale_price = (df["Sale_Price"] <= 0).sum()
sale_above_original = (df["Sale_Price"] > df["Original_Price"]).sum()

print("Original price <= 0:", invalid_original_price)
print("Sale price <= 0:", invalid_sale_price)
print("Sale price higher than original price:", sale_above_original)

Original price <= 0: 0
Sale price <= 0: 0
Sale price higher than original price: 0


## 5. Clean text and date columns

I make text values consistent and convert date columns into proper date format.

This is useful because inconsistent spaces or capitalization can create separate categories in charts and PivotTables.

In [9]:
# I remove extra spaces from important text columns.

text_columns = [
    "Product_ID",
    "Brand",
    "Product_Name",
    "SKU",
    "Category",
    "Subcategory",
    "Collection",
    "Availability",
    "Currency",
    "Product_URL",
    "Source_Website",
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# I convert the date columns into pandas date values.
for column in ["Snapshot_Date", "Scrape_Date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

print("Text and date columns cleaned.")

Text and date columns cleaned.


## 6. Clean numeric columns

I convert the price and discount fields into numeric values.

I then recalculate the discount amount and percentage from the prices. This gives me one consistent calculation for all three brands.

In [10]:
# I convert my price fields to numeric values.

price_columns = [
    "Original_Price",
    "Sale_Price",
    "Discount_Amount",
    "Discount_Percent",
]

for column in price_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# I remove exact duplicate product URLs if any appeared during scraping.
df = (
    df.drop_duplicates(subset=["Brand", "Product_URL"])
      .reset_index(drop=True)
)

# I keep only rows with valid positive prices.
df = df[
    (df["Original_Price"] > 0)
    & (df["Sale_Price"] > 0)
].copy()

# If a sale price is above the original price, I use the sale price as the original price.
df["Original_Price"] = np.maximum(
    df["Original_Price"],
    df["Sale_Price"]
)

# I recalculate discount fields from the cleaned prices.
df["Discount_Amount"] = (
    df["Original_Price"] - df["Sale_Price"]
).round(2)

df["Discount_Percent"] = np.where(
    df["Original_Price"] > 0,
    (df["Discount_Amount"] / df["Original_Price"]) * 100,
    0
)

df["Discount_Percent"] = df["Discount_Percent"].round(2)

print("Numeric fields cleaned and discounts recalculated.")

Numeric fields cleaned and discounts recalculated.


## 7. Standardize categories

The three brand websites do not always use exactly the same naming style. I standardize the main project categories so they work properly in my final dashboard.

In [11]:
# I keep one standard name for each main project category.

category_map = {
    "unstitched": "Unstitched",
    "pret": "Pret",
    "ready to wear": "Pret",
    "ready-to-wear": "Pret",
    "stitched": "Stitched",
    "luxury pret": "Luxury Pret",
    "formal": "Formal",
    "formals": "Formal",
}

df["Category"] = (
    df["Category"]
    .str.lower()
    .map(category_map)
    .fillna(df["Category"])
)

print("Final categories:")
print(sorted(df["Category"].unique()))

Final categories:
['Formal', 'Luxury Pret', 'Pret', 'Stitched', 'Unstitched']


## 8. Create Sale Status

I divide products into two simple groups:

- **On Sale** — sale price is lower than original price.
- **Regular Price** — there is no price reduction.

This field will be useful as an interactive dashboard filter.

In [12]:
# I create a simple sale-status field from the cleaned prices.

df["Sale_Status"] = np.where(
    df["Sale_Price"] < df["Original_Price"],
    "On Sale",
    "Regular Price"
)

df["Sale_Status"].value_counts()

Sale_Status
Regular Price    242
On Sale           39
Name: count, dtype: int64

## 9. Create Price Segment

For this project, I use four price groups based on the current product range:

- **Budget:** below PKR 8,000
- **Mid Range:** PKR 8,000 to 14,999
- **Premium:** PKR 15,000 to 29,999
- **Luxury:** PKR 30,000 and above

These groups make the price-positioning chart easier to understand.

In [13]:
# I create four price segments using the current sale price.

price_conditions = [
    df["Sale_Price"] < 8000,
    df["Sale_Price"].between(8000, 14999.99),
    df["Sale_Price"].between(15000, 29999.99),
    df["Sale_Price"] >= 30000,
]

price_labels = [
    "Budget",
    "Mid Range",
    "Premium",
    "Luxury",
]

df["Price_Segment"] = np.select(
    price_conditions,
    price_labels,
    default="Unknown"
)

df["Price_Segment"].value_counts()

Price_Segment
Mid Range    82
Premium      76
Budget       74
Luxury       49
Name: count, dtype: int64

## 10. Create Discount Band

I also group discounts so I can show the discount strategy more clearly in Excel.

In [14]:
# I group products by discount percentage.

discount_conditions = [
    df["Discount_Percent"] == 0,
    df["Discount_Percent"].between(0.01, 19.99),
    df["Discount_Percent"].between(20, 39.99),
    df["Discount_Percent"].between(40, 59.99),
    df["Discount_Percent"] >= 60,
]

discount_labels = [
    "No Discount",
    "Low (1-19%)",
    "Medium (20-39%)",
    "High (40-59%)",
    "Very High (60%+)",
]

df["Discount_Band"] = np.select(
    discount_conditions,
    discount_labels,
    default="No Discount"
)

df["Discount_Band"].value_counts()

Discount_Band
No Discount         242
Low (1-19%)          15
High (40-59%)        11
Very High (60%+)      8
Medium (20-39%)       5
Name: count, dtype: int64

## 11. Create Category Price Index

A simple brand average can be unfair because one brand may have more formal or luxury products.

To make the price comparison more useful, I compare each product with the **median price of its own category**.

A Price Index:

- below **100** means the product is cheaper than the category median,
- around **100** means it is close to the category median,
- above **100** means it is more expensive than the category median.

In [15]:
# I calculate the median sale price for each main category.

category_median = (
    df.groupby("Category")["Sale_Price"]
      .transform("median")
)

df["Category_Median_Price"] = category_median.round(2)

# I calculate each product's price compared with its category median.
df["Price_Index"] = (
    (df["Sale_Price"] / df["Category_Median_Price"]) * 100
).round(2)

df[
    [
        "Brand",
        "Category",
        "Sale_Price",
        "Category_Median_Price",
        "Price_Index",
    ]
].head()

,Brand,Category,Sale_Price,Category_Median_Price,Price_Index
0,J.,Unstitched,6900.0,6900.0,100.00
1,J.,Unstitched,6900.0,6900.0,100.00
2,J.,Unstitched,6900.0,6900.0,100.00
3,J.,Unstitched,6900.0,6900.0,100.00
4,J.,Unstitched,4500.0,6900.0,65.22


## 12. Build the Brand Intelligence Score

I want one summary score for the dashboard, but I do not want it to be a random number.

I use four parts:

- **Price Competitiveness — 30%**
- **Discount Strategy — 25%**
- **Product Variety — 25%**
- **Availability — 20%**

For price competitiveness, a lower average Price Index gets a better score.

For discount strategy, I combine average discount and the share of products currently on sale.

For variety, I compare how many of the five project categories each brand covers.

For availability, I use the percentage of products marked available.

In [16]:
# I create one brand-level summary table for my intelligence score.

brand_metrics = (
    df.groupby("Brand")
      .agg(
          Product_Count=("Product_ID", "count"),
          Median_Price=("Sale_Price", "median"),
          Average_Price_Index=("Price_Index", "mean"),
          Average_Discount=("Discount_Percent", "mean"),
          Category_Count=("Category", "nunique"),
          Available_Products=(
              "Availability",
              lambda x: (x.str.lower() == "available").sum()
          ),
          Sale_Products=(
              "Sale_Status",
              lambda x: (x == "On Sale").sum()
          ),
      )
      .reset_index()
)

brand_metrics["Availability_Percent"] = (
    brand_metrics["Available_Products"]
    / brand_metrics["Product_Count"]
    * 100
)

brand_metrics["Sale_Product_Percent"] = (
    brand_metrics["Sale_Products"]
    / brand_metrics["Product_Count"]
    * 100
)

brand_metrics.round(2)

,Brand,Product_Count,Median_Price,Average_Price_Index,Average_Discount,Category_Count,Available_Products,Sale_Products,Availability_Percent,Sale_Product_Percent
0,J.,100,9990.0,85.96,1.70,3,100,4,100.0,4.00
1,Maria.B,100,23990.0,164.66,1.50,4,100,15,100.0,15.00
2,Sana Safinaz,81,8999.0,122.22,12.07,4,81,20,100.0,24.69


In [17]:
# I use a small min-max scoring function to convert different measures to a 0-100 scale.

def min_max_score(series, higher_is_better=True):
    series = series.astype(float)

    if series.max() == series.min():
        return pd.Series(
            [100.0] * len(series),
            index=series.index
        )

    score = (
        (series - series.min())
        / (series.max() - series.min())
        * 100
    )

    if not higher_is_better:
        score = 100 - score

    return score


# Lower category-adjusted prices receive a better price score.
brand_metrics["Price_Competitiveness_Score"] = min_max_score(
    brand_metrics["Average_Price_Index"],
    higher_is_better=False
)

# I combine average discount and sale-product share for the discount score.
discount_level_score = min_max_score(
    brand_metrics["Average_Discount"],
    higher_is_better=True
)

sale_share_score = min_max_score(
    brand_metrics["Sale_Product_Percent"],
    higher_is_better=True
)

brand_metrics["Discount_Strategy_Score"] = (
    discount_level_score * 0.60
    + sale_share_score * 0.40
)

# A brand covering more of my five project categories receives a better variety score.
total_project_categories = 5

brand_metrics["Product_Variety_Score"] = (
    brand_metrics["Category_Count"]
    / total_project_categories
    * 100
).clip(upper=100)

# Availability is already naturally measured as a percentage.
brand_metrics["Availability_Score"] = (
    brand_metrics["Availability_Percent"]
)

# I calculate the final Brand Intelligence Score out of 100.
brand_metrics["Brand_Intelligence_Score"] = (
    brand_metrics["Price_Competitiveness_Score"] * 0.30
    + brand_metrics["Discount_Strategy_Score"] * 0.25
    + brand_metrics["Product_Variety_Score"] * 0.25
    + brand_metrics["Availability_Score"] * 0.20
).round(2)

score_columns = [
    "Brand",
    "Price_Competitiveness_Score",
    "Discount_Strategy_Score",
    "Product_Variety_Score",
    "Availability_Score",
    "Brand_Intelligence_Score",
]

brand_metrics[score_columns].round(2)

,Brand,Price_Competitiveness_Score,Discount_Strategy_Score,Product_Variety_Score,Availability_Score,Brand_Intelligence_Score
0,J.,100.00,1.14,60.0,100.0,65.28
1,Maria.B,0.00,21.26,80.0,100.0,45.32
2,Sana Safinaz,53.93,100.00,80.0,100.0,81.18


## 13. Add the brand scores to my processed dataset

I merge the brand-level score fields back into the product dataset.

This makes it easier to use the same processed CSV later in PostgreSQL and Excel.

In [18]:
# I merge the brand intelligence fields into every product row.

score_fields = brand_metrics[
    [
        "Brand",
        "Price_Competitiveness_Score",
        "Discount_Strategy_Score",
        "Product_Variety_Score",
        "Availability_Score",
        "Brand_Intelligence_Score",
    ]
].copy()

df = df.merge(
    score_fields,
    on="Brand",
    how="left"
)

print("Brand scores added to the processed dataset.")

Brand scores added to the processed dataset.


## 14. Final quality check

Before exporting, I check the final row count, missing values, duplicates, and a simple brand summary.

In [19]:
# I check the final size of my processed dataset.

print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Duplicate product URLs:", df.duplicated(["Brand", "Product_URL"]).sum())
print("Total missing values:", df.isna().sum().sum())

Final rows: 281
Final columns: 27
Duplicate product URLs: 0
Total missing values: 0


In [20]:
# I prepare a simple final summary that I can later compare with my dashboard.

final_summary = (
    df.groupby("Brand")
      .agg(
          Products=("Product_ID", "count"),
          Median_Price=("Sale_Price", "median"),
          Average_Discount=("Discount_Percent", "mean"),
          Categories=("Category", "nunique"),
          Sale_Products=("Sale_Status", lambda x: (x == "On Sale").sum()),
          Brand_Intelligence_Score=("Brand_Intelligence_Score", "first"),
      )
      .reset_index()
)

final_summary.round(2)

,Brand,Products,Median_Price,Average_Discount,Categories,Sale_Products,Brand_Intelligence_Score
0,J.,100,9990.0,1.70,3,4,65.28
1,Maria.B,100,23990.0,1.50,4,15,45.32
2,Sana Safinaz,81,8999.0,12.07,4,20,81.18


## 15. Save the processed dataset

This processed CSV will be my main analytical dataset.

I will use the same file in the next project steps instead of creating different versions of the data.

In [21]:
# I arrange the most important columns in a clear order before saving.

preferred_order = [
    "Product_ID",
    "Snapshot_Date",
    "Brand",
    "Product_Name",
    "SKU",
    "Category",
    "Subcategory",
    "Collection",
    "Original_Price",
    "Sale_Price",
    "Discount_Amount",
    "Discount_Percent",
    "Sale_Status",
    "Price_Segment",
    "Discount_Band",
    "Category_Median_Price",
    "Price_Index",
    "Availability",
    "Currency",
    "Price_Competitiveness_Score",
    "Discount_Strategy_Score",
    "Product_Variety_Score",
    "Availability_Score",
    "Brand_Intelligence_Score",
    "Product_URL",
    "Source_Website",
    "Scrape_Date",
]

df = df[preferred_order]

# I save the cleaned and prepared dataset for PostgreSQL and Excel.
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Processed dataset saved successfully.")
print("File name:", OUTPUT_FILE)

Processed dataset saved successfully.
File name: 02_processed_fashion_products.csv


## 16. Step 2 conclusion

In this notebook, I cleaned my raw web-scraped dataset and prepared the analytical fields needed for the project.

My main new fields are:

- Sale Status
- Price Segment
- Discount Band
- Category Median Price
- Price Index
- Price Competitiveness Score
- Discount Strategy Score
- Product Variety Score
- Availability Score
- Brand Intelligence Score

The output of this step is:

**`02_processed_fashion_products.csv`**

### Next step

In Step 3, I will load this processed dataset into **PostgreSQL** and answer business questions using SQL queries such as:

- median and average price by brand,
- discount strategy by brand,
- category mix,
- price segment distribution,
- top discounted products,
- and brand ranking.